# Autoencoder from Scratch — TensorFlow

**What you will learn in this notebook:**
1. What an autoencoder is and why it works
2. How to build an Encoder that compresses data
3. What the Latent Space actually contains
4. How to build a Decoder that reconstructs data
5. What Reconstruction Loss measures
6. How to visualise and interpret results
7. A practical application: anomaly detection

**Dataset:** MNIST handwritten digits (28×28 grayscale images)  
**Framework:** TensorFlow 2.x / Keras  
**Prerequisite:** You understand a basic dense neural network

---

## The Big Picture

An autoencoder learns to:
```
Input (784 pixels)
  → COMPRESS → Latent vector (32 numbers)   ← encoder
  → EXPAND   → Output (784 pixels)          ← decoder
```
The training objective is simple: **make the output look like the input.**

The network is forced to learn a *compact* representation because the latent
space (32 dims) is much smaller than the input (784 dims). It's like fitting
a photo through a tiny pipe — you must discard the noise and keep only the essence.

## Cell 1 — Imports

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt

print('TensorFlow version:', tf.__version__)
print('Keras version     :', keras.__version__)

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

## Cell 2 — Load & Inspect Data

MNIST has 70,000 handwritten digit images (0–9), each 28×28 pixels.

**Key preprocessing decisions:**
- Normalise pixels to [0, 1] → keeps activations and loss values stable
- Flatten 28×28 → 784 → works with Dense (fully-connected) layers
- **Discard labels** → autoencoders are unsupervised, no labels needed

In [ ]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

print('Before preprocessing:')
print(f'  x_train: {x_train.shape}, dtype: {x_train.dtype}, range: [{x_train.min()}, {x_train.max()}]')

# Normalise to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test  = x_test.astype('float32')  / 255.0

# Flatten: (N, 28, 28) → (N, 784)
x_train = x_train.reshape(-1, 784)
x_test  = x_test.reshape(-1, 784)

print('\nAfter preprocessing:')
print(f'  x_train: {x_train.shape}, range: [{x_train.min():.1f}, {x_train.max():.1f}]')
print(f'  x_test:  {x_test.shape}')

# Visualise a few samples
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for i, ax in enumerate(axes):
    ax.imshow(x_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(str(y_train[i]), fontsize=10)
    ax.axis('off')
plt.suptitle('Sample training images (label shown above)', y=1.05)
plt.tight_layout()
plt.show()

## Cell 3 — Build the Encoder

The encoder is a funnel:
```
784 → 256 → 128 → 64 → 32
```
Each layer has fewer neurons than the last, forcing the network to learn
which features are *essential* and which can be discarded.

**Why no activation on the last encoder layer?**  
The latent vector should be free to take any value. If we applied sigmoid,
values would be squashed to [0,1] which restricts what the decoder can work with.

In [ ]:
LATENT_DIM = 32   # change to 2 to visualise the latent space as a 2D scatter plot
INPUT_DIM  = 784

def build_encoder(input_dim, latent_dim):
    inputs = keras.Input(shape=(input_dim,), name='encoder_input')
    
    # Progressive compression
    x = layers.Dense(256, activation='relu', name='enc_1')(inputs)
    x = layers.Dense(128, activation='relu', name='enc_2')(x)
    x = layers.Dense(64,  activation='relu', name='enc_3')(x)
    
    # Bottleneck — no activation, values are free
    latent = layers.Dense(latent_dim, name='latent_vector')(x)
    
    return Model(inputs, latent, name='encoder')

encoder = build_encoder(INPUT_DIM, LATENT_DIM)
encoder.summary()

# Quick test: feed one image through the encoder
test_input  = x_test[:1]                          # shape: (1, 784)
test_latent = encoder.predict(test_input, verbose=0)  # shape: (1, 32)
print(f'\nInput shape  : {test_input.shape}')
print(f'Latent shape : {test_latent.shape}')
print(f'Latent vector: {np.round(test_latent[0], 3)}')

## Cell 4 — Understand the Latent Space

The latent space is the **compressed memory** of the autoencoder.

- Each image becomes a single point in 32-dimensional space
- Similar images map to nearby points
- The network must pack ALL information about the digit into 32 numbers

Let's look at what the encoder produces for different digit classes.

In [ ]:
# Encode a few examples from each digit class
# (encoder is untrained yet — this shows what random latent vectors look like)
print('Latent vectors for first 5 test images (untrained encoder):')
print('Each row = one image compressed to', LATENT_DIM, 'numbers')
print()

for i in range(5):
    z = encoder.predict(x_test[i:i+1], verbose=0)[0]
    print(f'  Digit {y_test[i]}: min={z.min():.3f}, max={z.max():.3f}, mean={z.mean():.3f}')

print()
print('After training, images of the same digit should produce similar latent vectors.')
print('Right now they are random — the encoder has not learned anything yet.')

## Cell 5 — Build the Decoder

The decoder is the mirror-image of the encoder — an inverse funnel:
```
32 → 64 → 128 → 256 → 784
```

**Why sigmoid on the output layer?**  
Pixel values were normalised to [0, 1]. Sigmoid guarantees the output
is also in [0, 1], so it can be directly compared to the input.

In [ ]:
def build_decoder(latent_dim, output_dim):
    latent_inputs = keras.Input(shape=(latent_dim,), name='decoder_input')
    
    # Progressive expansion (mirror of encoder)
    x = layers.Dense(64,  activation='relu', name='dec_1')(latent_inputs)
    x = layers.Dense(128, activation='relu', name='dec_2')(x)
    x = layers.Dense(256, activation='relu', name='dec_3')(x)
    
    # Output: sigmoid to keep values in [0, 1] (same range as input)
    outputs = layers.Dense(output_dim, activation='sigmoid', name='reconstruction')(x)
    
    return Model(latent_inputs, outputs, name='decoder')

decoder = build_decoder(LATENT_DIM, INPUT_DIM)
decoder.summary()

# Quick test: decode a random latent vector
random_latent = np.random.randn(1, LATENT_DIM).astype('float32')
decoded       = decoder.predict(random_latent, verbose=0)
print(f'\nRandom latent shape : {random_latent.shape}')
print(f'Decoded shape       : {decoded.shape}')
print(f'Decoded value range : [{decoded.min():.3f}, {decoded.max():.3f}]')

## Cell 6 — Reconstruction Loss (the math)

Before training, let's understand what the loss actually measures.

**Mean Squared Error (MSE):**
```
MSE = (1/784) × Σ (original_pixel - reconstructed_pixel)²
```

- Perfect reconstruction → MSE = 0.0
- Random reconstruction  → MSE ≈ 0.167  (random uniform vs uniform input)
- Trained model on MNIST → MSE ≈ 0.01–0.02 (very good)

The gradient flows through the decoder and encoder, teaching both.

In [ ]:
# Manual MSE calculation — exact same thing Keras computes
original     = x_test[0]                    # shape: (784,)
perfect_recon = original.copy()             # identical copy
random_recon  = np.random.rand(784).astype('float32')  # garbage

def mse(a, b):
    return float(np.mean((a - b) ** 2))

print('Reconstruction Loss (MSE) examples:')
print(f'  Perfect reconstruction : {mse(original, perfect_recon):.6f}')
print(f'  Random reconstruction  : {mse(original, random_recon):.6f}')
print(f'  All-zeros output       : {mse(original, np.zeros(784)):.6f}')
print(f'  All-0.5 output         : {mse(original, np.full(784, 0.5)):.6f}')
print()
print('Goal: train the network so MSE is as low as possible.')

## Cell 7 — Assemble the Full Autoencoder

Connect encoder + decoder into one end-to-end model.

**The critical insight about training:**
```python
model.fit(x_train, x_train)   # input = target
```
The model tries to output what was fed in. This is **self-supervised learning**.

In [ ]:
# Wire encoder → decoder
ae_input  = keras.Input(shape=(INPUT_DIM,), name='ae_input')
encoded   = encoder(ae_input)
decoded   = decoder(encoded)

autoencoder = Model(ae_input, decoded, name='autoencoder')

autoencoder.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse',
    metrics=['mae'],
)

autoencoder.summary()

# Count parameters
total_params = autoencoder.count_params()
print(f'\nTotal trainable parameters: {total_params:,}')

## Cell 8 — Train the Autoencoder

Training typically takes 5–10 minutes on CPU.  
On GPU it takes ~1 minute.

**What to watch:**
- `loss` should decrease each epoch
- `val_loss` should decrease alongside it
- If `val_loss` stops improving, `EarlyStopping` will halt training

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

history = autoencoder.fit(
    x_train, x_train,       # <-- input = target (self-supervised!)
    epochs=30,
    batch_size=256,
    validation_data=(x_test, x_test),
    callbacks=callbacks,
    verbose=1,
)

print(f'\nFinal train MSE : {history.history["loss"][-1]:.6f}')
print(f'Final val   MSE : {history.history["val_loss"][-1]:.6f}')
print(f'Best val    MSE : {min(history.history["val_loss"]):.6f}')

## Cell 9 — Plot Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, len(history.history['loss']) + 1)

ax1.plot(epochs_range, history.history['loss'],     label='Train MSE', linewidth=2)
ax1.plot(epochs_range, history.history['val_loss'], label='Val MSE',   linewidth=2, linestyle='--')
ax1.set_title('Reconstruction Loss (MSE)\nLower = better reconstruction')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('MSE')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history.history['mae'],     label='Train MAE', linewidth=2)
ax2.plot(epochs_range, history.history['val_mae'], label='Val MAE',   linewidth=2, linestyle='--')
ax2.set_title('Mean Absolute Error\n(easier to interpret: avg pixel error)')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretation: if train and val loss are close, the model generalises well.')

## Cell 10 — Visualise Reconstructions

The most important sanity check: do the reconstructions look like the originals?

In [ ]:
n = 10
indices = np.random.choice(len(x_test), n, replace=False)
samples = x_test[indices]
reconstructed = autoencoder.predict(samples, verbose=0)

fig, axes = plt.subplots(2, n, figsize=(n * 1.5, 3.5))
fig.suptitle('Top: Original    Bottom: Reconstructed', fontsize=12, y=1.02)

for i in range(n):
    axes[0, i].imshow(samples[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[0, i].axis('off')

    axes[1, i].imshow(reconstructed[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
    axes[1, i].axis('off')
    
    mse_val = float(np.mean((samples[i] - reconstructed[i]) ** 2))
    axes[1, i].set_xlabel(f'MSE={mse_val:.4f}', fontsize=7)

plt.tight_layout()
plt.savefig('reconstructions.png', dpi=150, bbox_inches='tight')
plt.show()

overall_mse = float(np.mean((x_test[:1000] - autoencoder.predict(x_test[:1000], verbose=0)) ** 2))
print(f'Overall reconstruction MSE on 1000 test images: {overall_mse:.6f}')

## Cell 11 — Explore the Latent Space

What did the encoder actually learn?  
Let's look at the distribution of latent values across digits.

In [ ]:
# Encode all test images
z_test = encoder.predict(x_test, verbose=0)  # shape: (10000, 32)
print(f'Latent space shape: {z_test.shape}')
print(f'Value range: [{z_test.min():.3f}, {z_test.max():.3f}]')
print(f'Mean: {z_test.mean():.3f}, Std: {z_test.std():.3f}')
print()

# Compare latent vectors for two images of the same digit
zeros = x_test[y_test == 0][:2]
z0, z1 = encoder.predict(zeros, verbose=0)
similarity = np.dot(z0, z1) / (np.linalg.norm(z0) * np.linalg.norm(z1))
print(f'Cosine similarity between two "0" digits: {similarity:.4f}')

# Compare latent vectors for images of different digits
zero_img = x_test[y_test == 0][:1]
one_img  = x_test[y_test == 1][:1]
z_zero = encoder.predict(zero_img, verbose=0)[0]
z_one  = encoder.predict(one_img,  verbose=0)[0]
diff_similarity = np.dot(z_zero, z_one) / (np.linalg.norm(z_zero) * np.linalg.norm(z_one))
print(f'Cosine similarity between "0" and "1" digits: {diff_similarity:.4f}')
print()
print('If trained well: same-digit similarity > different-digit similarity')

## Cell 12 — 2D Latent Space Visualisation

Re-run with `LATENT_DIM = 2` to see this clearly.  
With 32 dims, we use PCA to project down to 2D for visualisation.

In [ ]:
from sklearn.decomposition import PCA

z_test = encoder.predict(x_test, verbose=0)

if z_test.shape[1] == 2:
    z_2d = z_test
    title = '2D Latent Space (direct)'
else:
    # Project to 2D via PCA for visualisation
    pca = PCA(n_components=2)
    z_2d = pca.fit_transform(z_test)
    explained = pca.explained_variance_ratio_.sum() * 100
    title = f'Latent Space — PCA projection ({explained:.1f}% variance)'

plt.figure(figsize=(9, 7))
scatter = plt.scatter(z_2d[:, 0], z_2d[:, 1],
                      c=y_test, cmap='tab10', alpha=0.4, s=3)
plt.colorbar(scatter, label='Digit class')
plt.title(title)
plt.xlabel('Component 1')
plt.ylabel('Component 2')
plt.tight_layout()
plt.savefig('latent_space.png', dpi=150, bbox_inches='tight')
plt.show()

print('What to look for:')
print('  Distinct clusters = encoder learned to separate digit classes')
print('  Cluster overlap   = encoder finds these digits similar (e.g. 3 and 8)')

## Cell 13 — Latent Space Interpolation

Walk from one digit to another through the latent space.  
This is only possible because the latent space is **continuous**.

In [ ]:
def interpolate(encoder, decoder, x_test, y_test, digit_a=0, digit_b=1, steps=10):
    idx_a = np.where(y_test == digit_a)[0][0]
    idx_b = np.where(y_test == digit_b)[0][0]

    z_a = encoder.predict(x_test[idx_a:idx_a+1], verbose=0)
    z_b = encoder.predict(x_test[idx_b:idx_b+1], verbose=0)

    alphas  = np.linspace(0, 1, steps)
    z_interp = np.array([z_a + a * (z_b - z_a) for a in alphas]).squeeze()
    images   = decoder.predict(z_interp, verbose=0)

    fig, axes = plt.subplots(1, steps, figsize=(steps * 1.5, 2))
    fig.suptitle(f'Latent interpolation: {digit_a} → {digit_b}', fontsize=11)
    for i, ax in enumerate(axes):
        ax.imshow(images[i].reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        ax.set_title(f'{alphas[i]:.1f}', fontsize=7)
    plt.tight_layout()
    plt.savefig(f'interp_{digit_a}_{digit_b}.png', dpi=150, bbox_inches='tight')
    plt.show()

interpolate(encoder, decoder, x_test, y_test, digit_a=0, digit_b=1)
interpolate(encoder, decoder, x_test, y_test, digit_a=3, digit_b=8)

## Cell 14 — Anomaly Detection

A practical application of autoencoders.

**Idea:** Train on "normal" data (digits 0–4). The model learns to reconstruct
these well. When given "anomalous" data (digits 5–9), it can't reconstruct them
as well → higher MSE → flag as anomaly.

This generalises to: fraud detection, medical anomalies, manufacturing defects.

In [ ]:
# --- Simulate: digits 0-4 are 'normal', 5-9 are 'anomalies' ---
normal_mask  = y_test <= 4
anomaly_mask = y_test >= 5

x_normal  = x_test[normal_mask][:500]
x_anomaly = x_test[anomaly_mask][:500]

# Reconstruction loss per sample
def per_sample_mse(model, data):
    recon = model.predict(data, verbose=0)
    return np.mean((data - recon) ** 2, axis=1)

losses_normal  = per_sample_mse(autoencoder, x_normal)
losses_anomaly = per_sample_mse(autoencoder, x_anomaly)

# Set threshold at 95th percentile of normal losses
threshold = np.percentile(losses_normal, 95)
print(f'Threshold (95th pctile of normal): {threshold:.6f}')
print(f'Normal   — mean MSE: {losses_normal.mean():.6f}')
print(f'Anomaly  — mean MSE: {losses_anomaly.mean():.6f}')

detected = (losses_anomaly > threshold).mean() * 100
print(f'\nAnomaly detection rate: {detected:.1f}%')

# Visualise the distribution of losses
plt.figure(figsize=(9, 4))
plt.hist(losses_normal,  bins=40, alpha=0.7, label='Normal (0-4)',   color='steelblue')
plt.hist(losses_anomaly, bins=40, alpha=0.7, label='Anomaly (5-9)',  color='coral')
plt.axvline(threshold, color='red', linewidth=2, linestyle='--',
            label=f'Threshold = {threshold:.4f}')
plt.title('Reconstruction Loss Distribution\nNormal vs Anomaly')
plt.xlabel('MSE (reconstruction loss)')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig('anomaly_detection.png', dpi=150, bbox_inches='tight')
plt.show()

## Cell 15 — Effect of Latent Dimension Size

How does the bottleneck size affect reconstruction quality?

In [ ]:
# Quick experiment: train tiny autoencoders with different latent dims
# NOTE: set to fewer epochs for speed here

latent_dims = [2, 8, 32, 64]
results     = {}

for ld in latent_dims:
    ae_input  = keras.Input(shape=(784,))
    x         = layers.Dense(128, activation='relu')(ae_input)
    z         = layers.Dense(ld)(x)                             # encoder
    x2        = layers.Dense(128, activation='relu')(z)         # decoder
    output    = layers.Dense(784, activation='sigmoid')(x2)
    mini_ae   = Model(ae_input, output)
    
    mini_ae.compile(optimizer='adam', loss='mse')
    mini_ae.fit(x_train, x_train, epochs=5, batch_size=256,
                validation_data=(x_test, x_test), verbose=0)
    
    val_loss = mini_ae.evaluate(x_test, x_test, verbose=0)[0]
    results[ld] = val_loss
    print(f'  latent_dim={ld:3d} → val MSE = {val_loss:.6f}')

plt.figure(figsize=(7, 4))
plt.plot(list(results.keys()), list(results.values()), 'o-', linewidth=2, markersize=8)
plt.xlabel('Latent dimension size')
plt.ylabel('Validation MSE')
plt.title('Reconstruction quality vs compression level\n(lower MSE = better reconstruction)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('latent_dim_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nObservation: larger latent_dim = better reconstruction, but less compression.')
print('The sweet spot depends on your application.')

## Summary — What You Built

| Component | What it does | Key detail |
|---|---|---|
| **Encoder** | 784 → 32 (compress) | No activation on last layer |
| **Latent space** | 32-dim representation | Smooth, continuous, structured |
| **Decoder** | 32 → 784 (reconstruct) | Sigmoid output for [0,1] pixels |
| **Recon. loss** | MSE(input, output) | Gradient teaches both halves |

## What's Next — VAE (Variational Autoencoder)

The main problem with a vanilla autoencoder:  
The latent space has **holes** — random points in it decode to garbage.

A VAE fixes this by making the latent space a **probability distribution** (Gaussian).  
Instead of encoding to a point `z`, it encodes to `(mean, variance)` and samples from it.  
This forces the latent space to be continuous and fully covered — you can generate
new data by sampling from it.

That's the subject of the next notebook.